# 03. 속성 분류 모델 학습

전처리에서 확정된 26개 속성 CSV를 그대로 사용해 `klue/roberta-base`를 파인튜닝합니다.

라벨 통합은 전처리 파일에서만 수행하며, 이 노트북은 확인·학습·평가만 담당합니다.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

root_candidates = [Path.cwd(), *Path.cwd().parents, Path("/home/bteam/aspect_sentiment")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "aspect_labels.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("aspect_labels.json을 찾을 수 없습니다. 프로젝트 폴더 안에서 실행하세요.")

settings = json.loads((PROJECT_ROOT / "aspect_labels.json").read_text(encoding="utf-8"))
MODEL_NAME = "klue/roberta-base"
MAX_LENGTH = 64
SEED = 42
DATA_DIR = PROJECT_ROOT / settings["data_dir"]
OUTPUT_DIR = PROJECT_ROOT / settings["model_output_dir"]
CHECKPOINT_DIR = Path("/tmp/aspect_sentiment_checkpoints/aspect")

set_seed(SEED)
print("프로젝트:", PROJECT_ROOT)
print("사용 장치:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
data = {
    split: pd.read_csv(DATA_DIR / f"{split}.csv")
    for split in ("train", "validation", "test")
}

expected_labels = {
    settings["rename_labels"].get(label, label)
    for label in settings["target_original_labels"]
    if label not in set(settings.get("exclude_labels", []))
}

for split, frame in data.items():
    missing = {"aspect", "sentiment_text"} - set(frame.columns)
    if missing:
        raise ValueError(f"{split} 필수 열 없음: {sorted(missing)}")
    frame["aspect"] = frame["aspect"].fillna("").astype(str).str.strip()
    frame["sentiment_text"] = (
        frame["sentiment_text"].fillna("").astype(str)
        .str.strip().str.replace(r"\s+", " ", regex=True)
    )
    if frame["sentiment_text"].eq("").any():
        raise ValueError(f"{split}에 빈 문장이 있습니다. 전처리를 다시 실행하세요.")

actual_train_labels = set(data["train"]["aspect"])
if actual_train_labels != expected_labels:
    missing = sorted(expected_labels - actual_train_labels)
    unknown = sorted(actual_train_labels - expected_labels)
    raise ValueError(f"최종 속성 불일치 - 없음: {missing}, 정의되지 않음: {unknown}")

for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = set(data[left]["sentiment_text"]) & set(data[right]["sentiment_text"])
    if overlap:
        raise ValueError(f"{left}-{right} 사이 중복 문장 {len(overlap)}개: 전처리를 다시 실행하세요.")

labels = sorted(expected_labels)
label2id = {label: number for number, label in enumerate(labels)}
id2label = {number: label for label, number in label2id.items()}
NUM_LABELS = len(labels)

for split, frame in data.items():
    unknown = sorted(set(frame["aspect"]) - set(label2id))
    if unknown:
        raise ValueError(f"{split}에 Train에 없는 속성 존재: {unknown}")
    frame["label"] = frame["aspect"].map(label2id).astype(int)

summary = pd.DataFrame({
    split: {
        "행 수": len(frame),
        "속성 수": frame["aspect"].nunique(),
        "카테고리 수": frame["category"].nunique() if "category" in frame else None,
    }
    for split, frame in data.items()
}).T
display(summary)
print("최종 속성 수:", NUM_LABELS)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["sentiment_text"], truncation=True, max_length=MAX_LENGTH)

datasets = {}
for split, frame in data.items():
    source = Dataset.from_pandas(
        frame[["sentiment_text", "label"]],
        preserve_index=False,
    )
    datasets[split] = source.map(
        tokenize,
        batched=True,
        remove_columns=["sentiment_text"],
    )

counts = data["train"]["label"].value_counts().sort_index()
weights = len(data["train"]) / (NUM_LABELS * counts.to_numpy())
class_weights = torch.tensor(weights, dtype=torch.float)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    label2id=label2id,
    id2label=id2label,
)

In [ ]:
def weighted_loss(outputs, labels, num_items_in_batch=None):
    return F.cross_entropy(
        outputs.logits,
        labels,
        weight=class_weights.to(outputs.logits.device),
    )

def compute_metrics(result):
    logits, answers = result
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = precision_recall_fscore_support(
        answers, predictions, average="macro", zero_division=0
    )[2]
    weighted_f1 = precision_recall_fscore_support(
        answers, predictions, average="weighted", zero_division=0
    )[2]
    return {
        "accuracy": accuracy_score(answers, predictions),
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    compute_loss_func=weighted_loss,
)

trainer.train()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

validation_metrics = trainer.evaluate(
    datasets["validation"],
    metric_key_prefix="validation",
)
test_output = trainer.predict(datasets["test"], metric_key_prefix="test")
test_predictions = np.argmax(test_output.predictions, axis=-1)

metrics = {
    "best_validation_macro_f1": trainer.state.best_metric,
    **validation_metrics,
    **test_output.metrics,
}
metrics_df = pd.DataFrame(metrics.items(), columns=["metric", "value"])
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False, encoding="utf-8-sig")

report = classification_report(
    data["test"]["label"],
    test_predictions,
    labels=list(range(NUM_LABELS)),
    target_names=[id2label[number] for number in range(NUM_LABELS)],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(OUTPUT_DIR / "classification_report.csv", encoding="utf-8-sig")

test_result = data["test"].copy()
test_result["predicted_label"] = test_predictions
test_result["predicted_aspect"] = test_result["predicted_label"].map(id2label)
wrong = test_result[test_result["label"] != test_result["predicted_label"]]
wrong.to_csv(
    OUTPUT_DIR / "test_wrong_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

display(metrics_df)
display(report_df.iloc[:NUM_LABELS].sort_values("f1-score"))
print("완료:", OUTPUT_DIR)
print("Test 오답:", len(wrong))